In [1]:
!pip install -q transformers accelerate bitsandbytes rank_bm25 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00


In [2]:
import pandas
df=pandas.read_csv('FraudData.csv')
df.head()
df.isnull().sum()

,0
LABEL,0
TEXT,0
URL,0
EMAIL,0
PHONE,0


In [3]:
pages = []
for idx, row in df.iterrows():
  page_text = (
        f"Classification Label: {row['LABEL']} | "
        f"Message Text: {row['TEXT']} | "
        f"Associated URL: {row['URL']}"
    )
  pages.append(page_text)
print(pages[0])

Classification Label: ham | Message Text: Your opinion about me? 1. Over 2. Jada 3. Kusruthi 4. Lovable 5. Silent 6. Spl character 7. Not matured 8. Stylish 9. Simple Pls reply.. | Associated URL: No


In [4]:
from rank_bm25 import BM25Okapi
tokenized_pages= [page.lower().split(" ") for page in pages]
bm25index = BM25Okapi(tokenized_pages)

In [5]:
def retrieve_relevant_pages(query, top_k=3):
  tokenized_query = query.lower().split(" ")
  top_pages = bm25index.get_top_n(tokenized_query,pages,n=top_k)
  return top_pages

In [6]:
test_query="urgent account suspended click the link"
results=retrieve_relevant_pages(test_query,top_k=2)
print(f"Top 2 matching records for our test query:\n")
for i, res in enumerate(results):
    print(f"--- Match {i+1} ---")
    print(res)
    print("-" * 20 + "\n")

Top 2 matching records for our test query:

--- Match 1 ---
Classification Label: Smishing | Message Text: XXXMobileMovieClub: To use your credit, click the WAP link in the next txt message or click here>> http://wap. xxxmobilemovieclub.com?n=QJKGIGHJJGCBL | Associated URL: yes
--------------------

--- Match 2 ---
Classification Label: spam | Message Text: XXXMobileMovieClub: To use your credit, click the WAP link in the next txt message or click here>> http://wap. xxxmobilemovieclub.com?n=QJKGIGHJJGCBL | Associated URL: yes
--------------------



In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [12]:
model_id="meta-llama/Llama-3.2-3B-Instruct"

In [13]:
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [22]:
"""from google.colab import userdata
hf_token=userdata.get('HF_TOKEN')"""

In [ ]:
!rm -rf ~/.cache/huggingface/
#not required for everyone, i was facing a problem so adde this one

In [ ]:
hf_token="hf_token" #add your own hugging face token here to access the hugging face repositpories
tokenizer=AutoTokenizer.from_pretrained(model_id,token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [29]:
def analyze_fraud(query_scenario):
  print(f"Analyzing the scenario: {query_scenario}")
  retrieve_context=retrieve_relevant_pages(query_scenario,top_k=3)
  context_str="\n".join([f"Record {i+1}:{ctx}" for i,ctx in enumerate(retrieve_context)])
  messages = [
        {"role": "system", "content": """You are an expert fraud and spam detection AI.
Analyze the suspicious activity using ONLY the provided dataset context.

You MUST output your response in the following exact format:
VERDICT: [Fraud / Safe / Spam / Phishing / Unknown]
CONFIDENCE LEVEL: [0-100]%
REASONING: [Your concise explanation based on the dataset matches]"""},
        {"role": "user", "content": f"Context from Dataset:\n{context_str}\n\nSuspicious Scenario to Analyze: {query_scenario}"}
    ]
  prompt=tokenizer.apply_chat_template(messages,tokenize=False, add_generation_prompt=True)
  inputs=tokenizer(prompt,return_tensors="pt").to("cuda")
  print("Analyszing....")
  outputs=model.generate(**inputs, max_new_tokens=300, temperature=0.1)
  response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
  return response

In [30]:
test_scenario = "I received an urgent text saying my bank account is restricted and I need to click a bit.ly link to verify my identity."
print("\n" + "="*50)
print("FINAL RAG PIPELINE TEST")
print("="*50)
final_result = analyze_fraud(test_scenario)
print("\nAI Analysis:")
print(final_result)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



FINAL RAG PIPELINE TEST
Analyzing the scenario: I received an urgent text saying my bank account is restricted and I need to click a bit.ly link to verify my identity.
Analyszing....

AI Analysis:
VERDICT: Phishing
CONFIDENCE LEVEL: 90%
REASONING: The message text suggests that the sender is claiming to be from the bank and is asking for the recipient to click a link to verify their identity, which is a common tactic used by scammers to trick victims into divulging sensitive information. The use of the phrase "urgent" and "account restricted" creates a sense of panic, which is often used to create a false sense of urgency and increase the likelihood of the recipient taking action without thinking critically. Additionally, the lack of a specific bank name or any other identifying information in the message text further suggests that this is a phishing attempt.
